## Exploring growth mechanics of apical hook maintenance phase of Col0 using GBVPy

Using finite element modeling (FEM), we investigated the biomechanics underlying apical hook morphogenesis by simulating growth patterns based on wild-type and aberrant Arabidopsis mutants.

In [ ]:
import sys
import os

from bvpy.domains import CustomDomain, CustomDomainGmsh
from bvpy.utils.pre_processing import HeterogeneousParameter
from bvpy.utils.io import save
import pyvista as pv
from bvpy.utils.visu_pyvista import visualize
from bvpy.vforms import LinearElasticForm, HyperElasticForm, StVenantKirchoffPotential
from bvpy.boundary_conditions import dirichlet, NormalDirichlet, NormalNeumann, neumann, Boundary
from bvpy.utils.visu import plot
import fenics as fe
import numpy as np
import sys
import meshio as mio
from bvpy import BVP
from subprocess import call, DEVNULL
import pandas as pd

from bvpy.gbvp import GBVP
from bvpy.vforms.plasticity import MorphoElasticForm, ConstantGrowth, StrainDependentGrowth, TimeDependentGrowth, HeterogeneousGrowth
from bvpy.utils.post_processing import SolutionExplorer


def xdmf_save(path, solution, vform, Fg, timestep):   
    """
    Save the displacement, strain, and stress fields to an XDMF file for a specific time step.

    Args:
        path (str): The file path to save the XDMF file.
        solution: The displacement vector solution.
        vform: Variational form object with strain and stress computation methods.
        Fg: Growth tensor for the current time step.
        time_step (float): The current simulation time step.
    """
    solution.rename("Displacement Vector", "")
    strain_e = vform.get_strain(solution, method='elastic', F_g=Fg)
    strain_t = vform.get_strain(solution, method='total', F_g=Fg)
    strain_e.rename("Elastic Strain", "")
    strain_t.rename("Total Strain", "")
    stress_e = vform.get_stress(solution, method='elastic', F_g=Fg)
    stress_t = vform.get_stress(solution, method='total', F_g=Fg)
    stress_e.rename("Elastic Stress", "")
    stress_t.rename("Total Stress", "")

    xdmf_file = fe.XDMFFile(fe.MPI.comm_world, path)
    xdmf_file.parameters["flush_output"] = True
    xdmf_file.parameters["functions_share_mesh"] = True
    xdmf_file.parameters["rewrite_function_mesh"] = False    
    xdmf_file.write(solution, timestep)
    xdmf_file.write(strain_e, timestep)
    xdmf_file.write(strain_t, timestep)
    xdmf_file.write(stress_e, timestep)
    xdmf_file.write(stress_t, timestep)
    
# Func for change mesh size
def generate_mesh(mesh_file, scale):
    """
    Save the displacement, strain, and stress fields to an XDMF file for a specific time step.
    Args:
        mesh_file (str): The file path to load the .geo file.
        scale (float): element size factor during mesh generation.
    """
    print("gmsh "+ mesh_file+ " -2"+ " -clscale "+ f'{scale}')
    call(["gmsh", mesh_file, "-2", "-clscale", f'{scale}'], stdout=DEVNULL, stderr=DEVNULL)


def hook_growth(hook_mesh, growth_param, name):
        """
    Apical hook growth simulation.
    1. Set the bc as locked hook base.
    2. Set the heterogeneous elastic properties
    3. Morpho Elastic Form with StVenantKirchoff model
    4. Establish heterogeneous Growth pattern with multiple growth scheme for each cell
    5. Define & solve the growth problem
    6. Save each time step of the simulation in a XDMF object
    Args:
        hook_mesh (CustomDomainGMSH): custom GMSH domain. hook base should be a line at min(y) 
        growth_param (pandas data frame): Table with Young_value per ID_label, and sig_xx, sig_xy, sig_yy of the growth tensor for each ID_label.
        name (str): prefix of the output file
    """
    # Boundary condition
    ymin = hook_mesh.mesh.coordinates()[:, 1].min()
    hook_base = Boundary(f'near(y, {ymin}, 1)')
    bc = dirichlet([0.,0.0, 0.0], boundary= hook_base)
    # Elastic properties
    young_values_by_labels = {row.ID_label: row.Young_value for _, row in growth_param.iterrows()}
    heterogeneous_young = HeterogeneousParameter(hook_mesh.cdata, young_values_by_labels)
    elastic_potential = StVenantKirchoffPotential(young=heterogeneous_young, poisson=0.4) # heterogeneous_young
    heterogeneous_Hyperelastic_response = HyperElasticForm(potential_energy=elastic_potential, source=[0., 0., 0.],
                                                       plane_stress=True)
    # Plastic vform
    plastic_vform = MorphoElasticForm(potential_energy=elastic_potential, F_g=np.identity(3), source = [0.,0.,0.])
    # Define the growth scheme
    growth_classes = {}
    growth_classes[0] = StrainDependentGrowth(extensibility=2., target_strain=0.01)
    for _, row in growth_param.iloc[1:].iterrows():
        id_label = int(row['ID_label'])
        x_growth = row['sig_xx']
        y_growth = row['sig_yy']
        xy_growth = row['sig_xy']
        # Create a ConstantGrowth object for each ID
        growth_classes[id_label] = ConstantGrowth(dg=np.array([[x_growth, xy_growth, 0.], 
                                                               [xy_growth, y_growth, 0], 
                                                               [0.,0.,0.]])
        )
    # Combine all growth scheme into a Heterogeneous Growth pattern
    growth_scheme = HeterogeneousGrowth(cdata = hook_mesh.cdata, growth_classes=growth_classes)
    # - Define problem
    growth_problem = GBVP(domain=hook_mesh, vform=plastic_vform, bc=bc, growth_scheme=growth_scheme)
    try:
        growth_problem.run(tmin=0, tmax=8, dt=0.1, store_steps=True, 
                   linear_solver='mumps', krylov_solver={'absolute_tolerance':1e-14}, 
                   relative_tolerance = 1e-6, absolute_tolerance = 1e-8,
                   preconditioner = 'none',
                   maximum_iterations = 500, line_search='bt')
    except Exception as e:
        print(f"Warning: growth.run() failed with error: {e}")
    
    time_points = list(growth_problem.solution_steps.keys())
    int_length = len(str(len(time_points)))+1
    for ix, t in enumerate(time_points):
        li = int(int_length-len(str(ix)))
        formatted_ix = ''.join(['0'] *  li)
        solution = growth_problem.solution_steps[t]
        Fg = growth_problem.growth_steps[t]
        save_path = f'./data/{name}_{formatted_ix}{ix}.xdmf'
        xdmf_save(save_path, solution, vform = plastic_vform, Fg= Fg, timestep = t)
    return growth_problem



## Generating the Mesh
Once the required libraries are loaded, the next step is to generate the mesh.

The `generate_mesh()` function uses GMSH to create .msh files from `.geo` files, which define the physical surface attributes.
The `CustomDomainGmsh()` class reads the `.msh` files and converts the data into a format compatible with **bvpy**.

In [ ]:
mesh_file = '../../data/hook_wall.geo'
scale = 2
generate_mesh(mesh_file, scale)
mesh_path = '../../data/hook_wall.msh'
cd = CustomDomainGmsh(fname = mesh_path)

### Visualizing the Mesh
To ensure the mesh has been correctly loaded and interpreted, you can visualize it using the following command:

In [ ]:
pl = visualize(cd, visu_type='domain', cmap='viridis')
pl.view_xy()

# Growth simulations

The `hook_growth` function simulates the **apical hook growth** using a finite element method (FEM) approach. It models the mechanical and morphoelastic behavior of the hook structure over time, incorporating **heterogeneous growth patterns** and **anisotropic material properties**. Below is a breakdown of the function's key steps:


### 1. Boundary Condition setup  
   - The **hook base** is locked in place, preventing movement along the bottom boundary.  
   - This ensures that deformation occurs **only in the upper parts** of the hook.

### 2. Heterogeneous elastic properties
   - Material stiffness is not uniform; different regions of the hook have varying **Young’s modulus** values.  
   - These values are extracted from `growth_param`, ensuring that each labeled region in `hook_mesh` has a unique mechanical response.

### 3. Morphoelastic formulation with St. Venant-Kirchhoff model
   - The **morphoelastic model** accounts for both **elastic deformations** and **growth-induced deformations**.  
   - The **St. Venant-Kirchhoff model** (a hyperelastic material model) is used to compute stress and strain distributions.

### 4. Heterogeneous growth pattern definition 
   - Different parts of the hook grow at different rates and in different directions.  
   - Growth is controlled by the **growth tensor**, extracted from `growth_param`.  
   - Multiple growth schemes are applied across the hook, ensuring that each cell in `hook_mesh` has its unique growth behavior.

### 5. Solving the growth problem
   - The FEM solver computes how the **hook deforms over time** under these above mention growth conditions.  
   - Growth progresses iteratively, updating the deformation at each time step.

### 6. Time-resolved data saving
   - The simulation results at each time step are saved in an **XDMF file format**, allowing visualization in **ParaView**.  
   - This enables post-processing, including analyzing displacement, stress, and strain evolution over time.

#### Summary
The growth simulation models apical hook morphogenesis with heterogeneous material properties and growth patterns. It locks the hook base, assigns different mechanical behaviors to regions, applies a morphoelastic formulation, solves the growth-induced deformation, and saves the simulation results for visualization in ParaView.


In [ ]:
growth_df = pd.read_csv("../../data/growth_data_ani.csv")  # growth data
growth_problem_heter_ani = hook_growth(hook_mesh = cd, growth_param = growth_df, name = 'Col0_heter_ani')

In [ ]:
growth_df = pd.read_csv("../../data/growth_data_iso.csv")
growth_problem_heter_iso = hook_growth(hook_mesh = cd, growth_param = growth_df, name = 'Col0__heter_iso')

In [ ]:
growth_df = pd.read_csv("../../data/growth_data_homog.csv")
growth_problem_homog_ani = hook_growth(hook_mesh = cd, growth_param = growth_df, name = 'Col0_homog_ani')

In [ ]:
growth_df = pd.read_csv("../../data/growth_data_homog_iso.csv")  # growth data
growth_problem_homog_iso = hook_growth(hook_mesh = cd, growth_param = growth_df, name = 'Col0_homog_iso')